## 1. PyTorch Geometric Installation

Detect the active PyTorch and CUDA versions, build the correct wheel URL, and install PyTorch Geometric and its compiled dependencies.

In [ ]:
import torch
import subprocess, sys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Enable GPU (2× T4) in Kaggle settings BEFORE running.'

print('Torch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)

print('\nInstalling PyTorch Geometric...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch-geometric'], check=True)

torch_v = torch.__version__.split('+')[0]
cuda_v = 'cu' + torch.version.cuda.replace('.', '')
wheel_url = f'https://data.pyg.org/whl/torch-{torch_v}+{cuda_v}.html'

print(f'\nInstalling PyG dependencies from {wheel_url}...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'pyg-lib', 'torch-scatter', 'torch-sparse',
                '-f', wheel_url], check=True)
print('\nInstallation complete.')


## 2. Repository Clone & Path Configuration

Clone the GitHub repo for access to source modules, configure paths to the attached Kaggle Datasets, and verify the GPU is active.

In [ ]:
import os, sys, torch

REPO_ROOT = '/kaggle/working/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
sys.path.insert(0, f'{REPO_ROOT}/src')

RAW_CACHE_DIR = '/kaggle/input/antenna-raw-cache-all-grids'
SEED_MASK_DIR = '/kaggle/input/antenna-seed-masks'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'GPU not available'
n_gpus = torch.cuda.device_count()
print(f'Device: {device} | GPUs: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')


## 3. GATv2 Attention Block

A single GATv2 convolution layer wrapped with LayerNorm, a residual connection (with optional linear projection when dimensions change), and ReLU activation. Two of these blocks form one "stage" of the network.

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv

class GATv2Block(nn.Module):
    def __init__(self, in_channels, out_channels, heads, edge_dim):
        super().__init__()
        self.conv = GATv2Conv(
            in_channels, out_channels // heads,
            heads=heads, edge_dim=edge_dim,
            concat=True, dropout=0.0
        )
        self.norm = nn.LayerNorm(out_channels)
        self.residual_proj = (nn.Linear(in_channels, out_channels)
                              if in_channels != out_channels else nn.Identity())
        self.act = nn.ReLU()
    
    def forward(self, x, edge_index, edge_attr):
        out = self.conv(x, edge_index, edge_attr=edge_attr)
        out = self.norm(out)
        out = self.act(out + self.residual_proj(x))
        return out


## 4. AntennaGNN Full Model

The complete GATv2-based surrogate model. It projects node and edge features, passes them through 4 blocks (8 GATv2 layers total), then combines a metal-only global mean pool with the virtual-node embedding to predict the 201-point S11 spectrum.

In [ ]:
from torch_geometric.nn import global_mean_pool

class AntennaGNN(nn.Module):
    def __init__(self, node_feat_dim=5, edge_feat_dim=2,
                 hidden_dim=128, heads=8, edge_dim=16,
                 num_blocks=4, output_dim=201):
        super().__init__()
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.edge_proj  = nn.Linear(edge_feat_dim, edge_dim)
        
        self.blocks = nn.ModuleList()
        for i in range(num_blocks):
            self.blocks.append(nn.ModuleList([
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim),
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim),
            ]))
        
        self.readout_proj = nn.Linear(hidden_dim * 2, 256)
        self.output_mlp = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(),
            nn.LayerNorm(512),
            nn.Linear(512, output_dim)
        )
    
    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        batch = data.batch
        
        x = self.input_proj(x)
        edge_attr = self.edge_proj(edge_attr)
        
        for block in self.blocks:
            for layer in block:
                x = layer(x, edge_index, edge_attr)
        
        # Metal-only pooling (metal = feature 0 of original input > 0.5)
        metal_mask = data.x[:, 0] > 0.5
        metal_x = x[metal_mask]
        metal_batch = batch[metal_mask]
        pooled = global_mean_pool(metal_x, metal_batch)  # (B, hidden_dim)
        
        # Virtual node embedding (is_seed == -1 flags the virtual node)
        virtual_mask = data.x[:, 3] == -1
        virtual_x = x[virtual_mask]  # (B, hidden_dim)
        
        combined = torch.cat([pooled, virtual_x], dim=-1)  # (B, 2*hidden_dim)
        out = self.readout_proj(combined)
        out = self.output_mlp(out)
        return out  # (B, 201)


## 5. Parameter Count Verification

Instantiate the model with default hyperparameters and verify that the total trainable parameter count falls within the expected range (800k–3M).

In [ ]:
model = AntennaGNN().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')
assert 500_000 <= total_params <= 3_000_000, f'Unexpected param count: {total_params}'
print('Parameter count OK.')


## 6. Build Sample Graphs from Raw Cache

Construct 3 test graphs directly from the raw `.pt` cache files (one each from the 25×25, 45×45, and 55×55 grids) to verify the full pipeline without depending on Chunk 5's processed output.

In [ ]:
import numpy as np
from torch_geometric.data import Data

def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)
    
    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])
    
    # Virtual global node: is_seed = -1 flags it for the model readout
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float32)
    
    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])
    
    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]
    
    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float32)
    y = torch.tensor(s11_db, dtype=torch.float32).unsqueeze(0)
    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)

samples = {}
for N in [25, 45, 55]:
    cache = torch.load(f'{RAW_CACHE_DIR}/raw_cache_{N}x{N}.pt', weights_only=False)
    seed_mask = np.load(f'{SEED_MASK_DIR}/seed_mask_{N}.npy')
    data = build_pyg_graph(cache['patterns'][0].numpy(),
                           cache['s11'][0].numpy(),
                           seed_mask, N)
    samples[N] = data
    print(f'Grid {N}x{N}: nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]}')
    del cache


## 7. Forward Pass Shape Verification

Batch the 3 sample graphs together and run a forward pass. Also test each grid size individually with batch_size=1 to ensure the model handles variable-sized graphs correctly.

In [ ]:
from torch_geometric.data import Batch

# Multi-graph batch test
batch = Batch.from_data_list([samples[25], samples[45], samples[55]]).to(device)
model.eval()
with torch.no_grad():
    out = model(batch)
print(f'Batch output shape: {out.shape}')
assert out.shape == (3, 201), f'Expected (3, 201), got {out.shape}'

# Single-graph tests per grid size
for N in [25, 45, 55]:
    single = Batch.from_data_list([samples[N]]).to(device)
    with torch.no_grad():
        out_single = model(single)
    assert out_single.shape == (1, 201), f'Grid {N}: expected (1, 201), got {out_single.shape}'
    print(f'Grid {N}x{N} single-graph: {out_single.shape} ✓')

print('\nAll shape checks PASSED ✓')


## 8. Gradient Flow Check

Run one forward + backward pass and verify that every trainable parameter received a non-None gradient, confirming there are no dead branches in the computation graph.

In [ ]:
model.train()
batch = Batch.from_data_list([samples[25], samples[45], samples[55]]).to(device)

out = model(batch)
loss = out.sum()  # dummy loss
loss.backward()

dead_params = []
for name, p in model.named_parameters():
    if p.requires_grad and p.grad is None:
        dead_params.append(name)

if dead_params:
    print('WARNING: Parameters with no gradient:')
    for n in dead_params:
        print(f'  {n}')
    raise RuntimeError(f'{len(dead_params)} parameters have no gradient!')
else:
    print(f'All {total_params:,} parameters received gradients ✓')


## 9. Write `model.py` for the Repository

Save the verified `GATv2Block` and `AntennaGNN` classes to `model.py` inside the cloned repo so it can be imported by downstream training notebooks.

In [ ]:
%%writefile /kaggle/working/antenna-gnn/src/model.py
"""
AntennaGNN — GATv2-based surrogate model for S11 spectrum prediction.

Architecture: 8 GATv2 layers (4 blocks × 2 layers), 8 attention heads,
128 hidden dim. Readout: metal-only global mean pool + virtual node
embedding → concat → MLP → 201-dim S11 spectrum.
"""

import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv, global_mean_pool


class GATv2Block(nn.Module):
    def __init__(self, in_channels, out_channels, heads, edge_dim):
        super().__init__()
        self.conv = GATv2Conv(
            in_channels, out_channels // heads,
            heads=heads, edge_dim=edge_dim,
            concat=True, dropout=0.0
        )
        self.norm = nn.LayerNorm(out_channels)
        self.residual_proj = (nn.Linear(in_channels, out_channels)
                              if in_channels != out_channels else nn.Identity())
        self.act = nn.ReLU()

    def forward(self, x, edge_index, edge_attr):
        out = self.conv(x, edge_index, edge_attr=edge_attr)
        out = self.norm(out)
        out = self.act(out + self.residual_proj(x))
        return out


class AntennaGNN(nn.Module):
    def __init__(self, node_feat_dim=5, edge_feat_dim=2,
                 hidden_dim=128, heads=8, edge_dim=16,
                 num_blocks=4, output_dim=201):
        super().__init__()
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.edge_proj  = nn.Linear(edge_feat_dim, edge_dim)

        self.blocks = nn.ModuleList()
        for i in range(num_blocks):
            self.blocks.append(nn.ModuleList([
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim),
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim),
            ]))

        self.readout_proj = nn.Linear(hidden_dim * 2, 256)
        self.output_mlp = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(),
            nn.LayerNorm(512),
            nn.Linear(512, output_dim)
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        batch = data.batch

        x = self.input_proj(x)
        edge_attr = self.edge_proj(edge_attr)

        for block in self.blocks:
            for layer in block:
                x = layer(x, edge_index, edge_attr)

        # Metal-only pooling
        metal_mask = data.x[:, 0] > 0.5
        metal_x = x[metal_mask]
        metal_batch = batch[metal_mask]
        pooled = global_mean_pool(metal_x, metal_batch)

        # Virtual node embedding
        virtual_mask = data.x[:, 3] == -1
        virtual_x = x[virtual_mask]

        combined = torch.cat([pooled, virtual_x], dim=-1)
        out = self.readout_proj(combined)
        out = self.output_mlp(out)
        return out


## 10. Display & Transfer Instructions

Print the saved `model.py` content for easy copy-paste, and provide instructions for transferring it back to the GitHub repository.

In [ ]:
!cat /kaggle/working/antenna-gnn/src/model.py

print('\n' + '='*70)
print('Kaggle sessions do not have git push credentials by default.')
print('Copy the model.py content printed above (or download it directly')
print('from the Kaggle Output file browser at antenna-gnn/src/model.py)')
print('and save it to REPO_ROOT/src/model.py in your local repo via')
print('Antigravity, then commit and push to GitHub so Chunk 7 can')
print('import it with: from model import AntennaGNN')
print('='*70)
